# Story brief v2: every computed figure, with its check

This notebook recomputes each figure marked "computed" in `briefs/STORY_BRIEF_v2.md` and checks it against the expected value given there. The notebook stops
on the first figure that does not reproduce: the text is never adjusted to fit the data. Results go to `results/` and the database, and a table of every
figure with its check status is written as `res_story_figure_checks`.

**Read this first.** These are descriptions of the data. The aluminium break-even is arithmetic on metal prices and two physical constants: it is not a forecast
and not a claim about what buyers do. Run `python copper_database/build_db.py --raw .` first.

In [1]:
import sqlite3
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
con = sqlite3.connect("../copper_database/copper.db")
rd = lambda sql: pd.read_sql(sql, con)
written, checks = {}, []


def save(df, name):
    df.to_csv(f"../results/{name}.csv", index=False)
    df.to_sql(name, con, if_exists="replace", index=False)
    written[name] = (len(df), len(df.columns))


def check(fig_id, figure, expected, computed, tol=0.0, unit=""):
    """Record one figure; stop the notebook if it does not reproduce."""
    if isinstance(expected, str):
        ok = str(computed) == expected
    else:
        ok = abs(float(computed) - float(expected)) <= tol
    checks.append({"figure_id": fig_id, "figure": figure, "expected_in_brief": expected, "computed": computed if isinstance(computed, str) else round(float(computed), 4),
                   "tolerance": tol, "unit": unit, "status": "PASS" if ok else "FAIL"})
    assert ok, f"{fig_id} does not reproduce: {figure}: brief says {expected}, the data gives {computed}"

## 1. Chapter "Uses": world copper use by end-use sector, and the single facts
ICSG World Copper Factbook 2025 (S31): shares of copper and copper-alloy semis use, world, 2024. The six shares are checked against the brief's order and labels.

In [2]:
eu = rd("SELECT sector, share_pct, scope, year, basis, source_id FROM copper_end_use")
facts = rd("SELECT fact_id, source_id, year, statement, value, unit, notes FROM copper_context_facts").set_index("fact_id")
order = ["Building construction", "Consumer and general products, cooling and electronics", "Infrastructure (power and telecom)", "Transport", "Industrial equipment", "Other"]
brief_shares = [26, 23, 17, 13, 12, 9]
eu = eu.set_index("sector").loc[order].reset_index()
for sec, exp, got in zip(order, brief_shares, eu.share_pct):
    check("U1", f"end-use share, {sec}", exp, got, 0, "percent")
check("U2", "the six end-use shares add up to 100", 100, eu.share_pct.sum(), 0, "percent")
check("U3", "buildings are the biggest single user", "Building construction", eu.sort_values("share_pct", ascending=False).sector.iloc[0])
check("U4", "wire share of first use (63%)", 63, facts.loc["F06", "value"], 0, "percent")
check("U5", "China's share of refined use (58%)", 58, facts.loc["F01", "value"], 0, "percent")
check("U6", "recycled share of use (about a third)", 33, facts.loc["F02", "value"], 0, "percent")
check("U7", "mine production 2024, million tonnes", 23, facts.loc["F05", "value"], 0, "Mt")
check("U8", "refined use 2024, million tonnes", 27.4, facts.loc["F04", "value"], 0, "Mt")
assert 60 <= facts.loc["F06", "value"] < 67, "the words 'nearly two thirds' need a wire share between 60 and 67 percent"
eu["rank"] = range(1, len(eu) + 1)
save(eu[["rank", "sector", "share_pct", "scope", "year", "basis", "source_id"]], "res_uses_end_use")
save(facts.reset_index(), "res_context_facts")
eu

,sector,share_pct,scope,year,basis,source_id,rank
0,Building construction,26,World,2024,share of copper and copper-alloy semis use,S31,1
1,"Consumer and general products, cooling and ele...",23,World,2024,share of copper and copper-alloy semis use,S31,2
2,Infrastructure (power and telecom),17,World,2024,share of copper and copper-alloy semis use,S31,3
3,Transport,13,World,2024,share of copper and copper-alloy semis use,S31,4
4,Industrial equipment,12,World,2024,share of copper and copper-alloy semis use,S31,5
5,Other,9,World,2024,share of copper and copper-alloy semis use,S31,6


## 2. Opener: the price over the last 12 months
World Bank monthly copper (S02), as quoted. August 2026 against August 2025.

In [3]:
px = rd("SELECT date, copper_usd_t FROM v_price_monthly ORDER BY date").set_index("date")["copper_usd_t"]
now_m, ago_m = px.index[-1], px.index[-13]
assert now_m == "2026-08-01" and ago_m == "2025-08-01"
chg = (px.iloc[-1] / px.loc[ago_m] - 1) * 100
check("O1", "copper price in August 2025 (USD per tonne)", 9670, px.loc[ago_m], 0, "USD/t")
check("O2", "copper price in August 2026 (USD per tonne)", 14326, px.iloc[-1], 0, "USD/t")
check("O3", "change over the last 12 months, shown as +48%", 48, round(chg), 0, "percent")
check("O4", "the exact 12-month change (+48.1%)", 48.1, chg, 0.05, "percent")
uf = pd.DataFrame([("copper_12m_change_pct", round(chg, 2), "percent", "August 2026 against August 2025, World Bank monthly average as quoted"),
                   ("copper_12m_ago_usd_t", float(px.loc[ago_m]), "USD per tonne", "August 2025"), ("copper_12m_ago_month", ago_m[:7], "month", ""),
                   ("copper_latest_usd_t", float(px.iloc[-1]), "USD per tonne", "August 2026")], columns=["fact_id", "value", "unit", "note"])
save(uf, "res_uses_facts")
uf

,fact_id,value,unit,note
0,copper_12m_change_pct,48.15,percent,"August 2026 against August 2025, World Bank mo..."
1,copper_12m_ago_usd_t,9670.0,USD per tonne,August 2025
2,copper_12m_ago_month,2025-08,month,
3,copper_latest_usd_t,14326.0,USD per tonne,August 2026


## 3. Chapter "Supply": mined here, refined there
USGS Mineral Commodity Summaries 2026 (S06), 2025 estimates. Mine share is of world mine output (23,000 kt); refinery share is of the rounded world refinery
output (29,000 kt). USGS lists no mine output for Japan, so Japan has no mine share (not zero).

In [4]:
mine = rd("SELECT country, value AS mine_kt FROM stg_usgs_copper WHERE statistic_detail = 'Mine production' AND year = 2025 AND section LIKE 'World Mine%' AND value IS NOT NULL")
ref = rd("SELECT country, value AS refinery_kt FROM stg_usgs_copper WHERE statistic_detail = 'Refinery production' AND year = 2025 AND value IS NOT NULL AND section LIKE 'World Mine%'")
w_mine = rd("SELECT value FROM stg_usgs_copper WHERE country = 'World total' AND statistic_detail = 'Mine production: rounded' AND year = 2025").value.iloc[0]
w_ref = rd("SELECT value FROM stg_usgs_copper WHERE country = 'World total' AND statistic_detail = 'Refinery production: rounded' AND year = 2025").value.iloc[0]
check("S1", "world mine output 2025 (kt)", 23000, w_mine, 0, "kt")
check("S2", "world refinery output 2025 (kt)", 29000, w_ref, 0, "kt")
names = rd("SELECT usgs_name AS country, display_name FROM usgs_country_iso")
want = [("China", 7.8, 48.3), ("Congo (Kinshasa)", 13.9, 9.7), ("Chile", 23.0, 5.9), ("Peru", 11.7, 1.2), ("Japan", None, 4.8), ("Russia", 5.7, 3.3), ("United States", 4.3, 2.9), ("Zambia", 4.1, 0.9)]
rv = pd.DataFrame({"country": [w[0] for w in want]}).merge(mine, on="country", how="left").merge(ref, on="country", how="left").merge(names, on="country", how="left")
rv["display_name"] = rv.display_name.fillna(rv.country)
rv["mine_share_pct"] = (rv.mine_kt / w_mine * 100).round(1)
rv["refinery_share_pct"] = (rv.refinery_kt / w_ref * 100).round(1)
rv["mine_listed"] = rv.mine_kt.notna().astype(int)
for (c, m, r), (_, row) in zip(want, rv.iterrows()):
    if m is None:
        check("S3", f"{c}: no mine output listed by USGS", "not listed", "not listed" if pd.isna(row.mine_kt) else "listed")
    else:
        check("S3", f"{c}: mine share of world output", m, row.mine_share_pct, 0.05, "percent")
    check("S4", f"{c}: refinery share of world output", r, row.refinery_share_pct, 0.05, "percent")
check("S5", "Chile mines 23% and refines 6% (rounded, as written)", "23/6", f"{round(rv.set_index('country').loc['Chile','mine_share_pct'])}/{round(rv.set_index('country').loc['Chile','refinery_share_pct'])}")
check("S6", "China mines 8% and refines 48% (rounded, as written)", "8/48", f"{round(rv.set_index('country').loc['China','mine_share_pct'])}/{round(rv.set_index('country').loc['China','refinery_share_pct'])}")
check("S7", "Chile and DR Congo mine 37% of the world's copper", 37, rv.set_index("country").loc[["Chile", "Congo (Kinshasa)"], "mine_share_pct"].sum(), 0.5, "percent")
check("S8", "China refines 48% (summary sentence)", 48, round(rv.set_index("country").loc["China", "refinery_share_pct"]), 0, "percent")
rv["world_mine_kt"] = w_mine
rv["world_refinery_kt"] = w_ref
rv = rv[["country", "display_name", "mine_kt", "mine_share_pct", "refinery_kt", "refinery_share_pct", "mine_listed", "world_mine_kt", "world_refinery_kt"]]
save(rv, "res_refined_vs_mined")
rv

,country,display_name,mine_kt,mine_share_pct,refinery_kt,refinery_share_pct,mine_listed,world_mine_kt,world_refinery_kt
0,China,China,1800.0,7.8,14000.0,48.3,1,23000.0,29000.0
1,Congo (Kinshasa),DR Congo,3200.0,13.9,2800.0,9.7,1,23000.0,29000.0
2,Chile,Chile,5300.0,23.0,1700.0,5.9,1,23000.0,29000.0
3,Peru,Peru,2700.0,11.7,340.0,1.2,1,23000.0,29000.0
4,Japan,Japan,NaN,NaN,1400.0,4.8,0,23000.0,29000.0
5,Russia,Russia,1300.0,5.7,950.0,3.3,1,23000.0,29000.0
6,United States,United States,1000.0,4.3,850.0,2.9,1,23000.0,29000.0
7,Zambia,Zambia,940.0,4.1,270.0,0.9,1,23000.0,29000.0


## 4. Chapter "Aluminium": the cheaper conductor metal
From NBS Circular 31 (S32): aluminium conducts 61% as well as copper, and weighs 2.70 against 8.89 g/cm3. For the same electrical job the aluminium wire needs
1 / 0.61 times the cross-section, and weighs (1 / 0.61) x (2.70 / 8.89) of the copper wire. Aluminium is the cheaper conductor metal whenever a tonne of copper
costs more than 1 / that mass share times a tonne of aluminium. Prices are World Bank monthly averages (S02); inflation cancels in a ratio of two prices in one currency.

In [5]:
pc = rd("SELECT material, property, value FROM physical_constants")
g = lambda mat, prop: pc[(pc.material.str.startswith(mat)) & (pc.property == prop)].value.iloc[0]
cond_al, dens_al, dens_cu = g("Aluminium", "volume_conductivity") / 100, g("Aluminium", "density_20C"), g("Copper", "density_20C")
area = 1 / cond_al
width = np.sqrt(area)
mass = area * dens_al / dens_cu
breakeven = 1 / mass
check("A1", "aluminium conducts 61% as well as copper", 0.61, cond_al, 0, "share")
check("A2", "cross-section needed (1.64 times)", 1.64, area, 0.005, "x")
check("A3", "the aluminium wire is about 28% wider", 28, (width - 1) * 100, 0.5, "percent")
check("A4", "mass of aluminium for the same job (0.50 of the copper)", 0.50, mass, 0.005, "x")
check("A5", "break-even price ratio (2.01)", 2.01, breakeven, 0.005, "ratio")
ratio = rd("SELECT month, cu_al_ratio FROM mart_cu_al_ratio ORDER BY month").set_index("month")["cu_al_ratio"]
latest = ratio.iloc[-1]
check("A6", "copper-to-aluminium ratio in August 2026 (4.41)", 4.41, latest, 0.005, "ratio")
share = mass / latest * 100
check("A7", "aluminium metal for the same conductance costs 11% of the copper metal", 11, round(share), 0, "percent")
check("A8", "that is about one ninth", 1 / 9 * 100, share, 1.0, "percent")
above = ratio > breakeven
last_below = above[~above].index.max()
first_all_above = above.loc[above.loc[last_below:].index[1:]].index.min() if last_below != ratio.index[-1] else None
months_after = ratio.loc[last_below:].iloc[1:]
check("A9", "every month since December 2008 is above the break-even: last month below is November 2008", "2008-11", last_below[:7])
check("A10", "the ratio in November 2008 (2.007)", 2.007, ratio.loc[last_below], 0.0005, "ratio")
check("A11", "all months after November 2008 are above", 0, int((months_after <= breakeven).sum()), 0, "months")
check("A12", "first month of the unbroken run is December 2008", "2008-12", months_after.index[0][:7])
check("A13", "share of months since 1960 above the break-even (42%)", 42, round(above.mean() * 100), 0, "percent")
n_run = len(months_after)
check("A14", "December 2008 to August 2026 is 17 years 9 months", "17 years 9 months", f"{n_run // 12} years {n_run % 12} months")
assert 17 < n_run / 12 < 18, "'almost 18 years' needs between 17 and 18 years"
case = pd.DataFrame([
    ("conductivity_vs_copper", cond_al, "share", "NBS Circular 31 (S32): hard-drawn aluminium against the annealed copper standard"),
    ("density_copper", dens_cu, "g/cm3", "S32"), ("density_aluminium", dens_al, "g/cm3", "S32"),
    ("area_factor", area, "x", "cross-section of aluminium wire per unit of copper wire, 1 / conductivity"),
    ("width_factor", width, "x", "diameter of the aluminium wire per unit of copper wire, square root of the area factor"),
    ("mass_factor", mass, "x", "mass of aluminium per unit of copper mass for the same electrical job"),
    ("breakeven_ratio", breakeven, "ratio", "aluminium is the cheaper conductor metal when copper costs more than this many times aluminium per tonne"),
    ("ratio_latest", latest, "ratio", "copper price divided by aluminium price, " + ratio.index[-1][:7]),
    ("cost_share_latest", share, "percent", "aluminium metal for the same conductance as a share of the copper metal's cost, latest month"),
    ("last_month_below", last_below[:7], "month", "last month with the ratio at or below the break-even"),
    ("last_below_ratio", float(ratio.loc[last_below]), "ratio", ""),
    ("first_month_of_run", months_after.index[0][:7], "month", "first month of the unbroken run above the break-even"),
    ("run_months", n_run, "months", "months from the first month of the run to the latest month, both included"),
    ("share_months_above", float(above.mean() * 100), "percent", "share of all months since January 1960 with the ratio above the break-even"),
    ("months_total", len(ratio), "months", "months in the ratio series"),
], columns=["fact_id", "value", "unit", "note"])
save(case, "res_aluminium_case")
case

,fact_id,value,unit,note
0,conductivity_vs_copper,0.61,share,NBS Circular 31 (S32): hard-drawn aluminium ag...
1,density_copper,8.89,g/cm3,S32
2,density_aluminium,2.7,g/cm3,S32
3,area_factor,1.639344,x,cross-section of aluminium wire per unit of co...
4,width_factor,1.280369,x,diameter of the aluminium wire per unit of cop...
5,mass_factor,0.497889,x,mass of aluminium per unit of copper mass for ...
6,breakeven_ratio,2.008481,ratio,aluminium is the cheaper conductor metal when ...
7,ratio_latest,4.406644,ratio,"copper price divided by aluminium price, 2026-08"
8,cost_share_latest,11.298588,percent,aluminium metal for the same conductance as a ...
9,last_month_below,2008-11,month,last month with the ratio at or below the brea...


## 5. Chapter "Demand": the Russia comparison
The reference total for 2030 (EV transition plus the data-centre extra over the 2024 build) against the USGS 2025 estimate of Russian mine output.

In [6]:
ref_total = rd("SELECT headline_total_t FROM res_demand_sensitivity WHERE bar_order = 0").headline_total_t.iloc[0] / 1000
russia = float(mine.set_index("country").loc["Russia", "mine_kt"])
check("D1", "reference total for 2030 (about 1.2 million tonnes)", 1189, round(ref_total), 0.5, "kt")
check("D2", "Russia mines 1,300 kt (2025e)", 1300, russia, 0, "kt")
gap = abs(ref_total - russia) / russia * 100
check("D3", "the total is within 10% of Russian mine output, so 'roughly' stands", "yes", "yes" if gap <= 10 else "no")
save(pd.DataFrame([("total_2030_kt", ref_total, "kt", "reference case, EV transition plus data-centre extra"), ("russia_mine_2025e_kt", russia, "kt", "USGS estimate"),
                   ("difference_pct", gap, "percent", "absolute difference as a share of Russian output"), ("within_10_pct", int(gap <= 10), "1 = yes", "so the page may say roughly")],
                  columns=["fact_id", "value", "unit", "note"]), "res_demand_scale")

## 6. The figure list
Every figure above with the value given in the brief, the value computed here and the result. The notebook has already stopped if any row failed.

In [7]:
fc = pd.DataFrame(checks)
assert (fc.status == "PASS").all()
save(fc, "res_story_figure_checks")
print(fc.status.value_counts().to_dict(), "figure checks")
fc

{'PASS': 56} figure checks


,figure_id,figure,expected_in_brief,computed,tolerance,unit,status
0,U1,"end-use share, Building construction",26,26.0,0.0000,percent,PASS
1,U1,"end-use share, Consumer and general products, ...",23,23.0,0.0000,percent,PASS
2,U1,"end-use share, Infrastructure (power and telecom)",17,17.0,0.0000,percent,PASS
3,U1,"end-use share, Transport",13,13.0,0.0000,percent,PASS
4,U1,"end-use share, Industrial equipment",12,12.0,0.0000,percent,PASS
5,U1,"end-use share, Other",9,9.0,0.0000,percent,PASS
6,U2,the six end-use shares add up to 100,100,100.0,0.0000,percent,PASS
7,U3,buildings are the biggest single user,Building construction,Building construction,0.0000,,PASS
8,U4,wire share of first use (63%),63,63.0,0.0000,percent,PASS
9,U5,China's share of refined use (58%),58,58.0,0.0000,percent,PASS


In [8]:
con.commit()
con.close()
print("Written (rows, columns):")
for k, v in written.items():
    print(f"  {k}: {v[0]} rows, {v[1]} columns")

Written (rows, columns):
  res_uses_end_use: 6 rows, 7 columns
  res_context_facts: 7 rows, 7 columns
  res_uses_facts: 4 rows, 4 columns
  res_refined_vs_mined: 8 rows, 9 columns
  res_aluminium_case: 15 rows, 4 columns
  res_demand_scale: 4 rows, 4 columns
  res_story_figure_checks: 56 rows, 7 columns
